# End-to-End FINN Flow for a Simple Convolutional Net
-----------------------------------------------------------------

In this notebook, we will go through the FINN steps needed to take a binarized convolutional network all the way down to a heterogeneous streaming dataflow accelerator running on the FPGA. 

It's recommended to go through the simpler [end-to-end notebook for a fully connected network](tfc_end2end_example.ipynb) first, since many steps here are very similar and we will focus on what is done differently for convolutions.

This notebook is quite lengthy, and some of the cells (involving Vivado synthesis) may take up to an hour to finish running. To let you save and resume your progress, we will save the intermediate ONNX models that are generated in the various steps to disk, so that you can jump back directly to where you left off.

In [1]:
from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
import os
    
#build_dir = os.environ["FINN_BUILD_DIR"]

## 1. Brevitas Export, FINN Import and Tidy-Up

Similar to what we did in the TFC-w1a1 end-to-end notebook, we will start by exporting the [pretrained CNV-w1a1 network](https://github.com/Xilinx/brevitas/tree/master/src/brevitas_examples/bnn_pynq) to ONNX, importing that into FINN and running the "tidy-up" transformations to have a first look at the topology. The network will be exported in QONNX format and then converted into the FINN-ONNX format to prepare it for the FINN compiler.

In [ ]:
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

export_onnx_path = "/home/teo/finn/quant_seq_unet_W4A4_b128_sigmoid.onnx"

qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_tidy.onnx")

### Adding Pre- and Postprocessing <a id='prepost'></a>

Preprocessing and postprocessing steps can be added directly in the ONNX graph. In this case, the preprocessing step divides the input `uint8` data by 255 so the inputs to the CNV-w1a1 network are bounded between [0, 1]. The postprocessing step takes the output of the network and returns the index (0-9) of the image category with the highest probability (top-1). 

In [3]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType

model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_tidy.onnx")
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
chkpt_preproc_name ="finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_preproc.onnx"
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

/home/teo/finn/deps/qonnx/src/qonnx/transformation/merge_onnx_models.py:70: UserWarning: [MergeONNXModels] opsets for models to merge differ: 14 vs 17, output model will use opset 17
  warnings.warn(
/home/teo/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [4]:
from qonnx.transformation.insert_topk import InsertTopK
from qonnx.transformation.infer_datatypes import InferDataTypes

# postprocessing: insert Top-1 node at the end
model = model.transform(InsertTopK(k=1))
chkpt_name = "finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_pre_post.onnx"
# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

## 2. How FINN Implements Convolutions: Lowering and Streamlining

In FINN, we implement convolutions with the *lowering* approach: we convert them to matrix-matrix multiply operations, where one of the matrices is generated by sliding a window over the input image. You can read more about the sliding window operator and how convolution lowering works [in this notebook](https://github.com/maltanar/qnn-inference-examples/blob/master/3-convolutional-binarized-gtsrb.ipynb). The streaming dataflow architecture we will end up with is going to look something like this figure from the [FINN-R paper](https://arxiv.org/abs/1809.04570):

![](cnv-mp-fc.png)

Note how the convolution layer looks very similar to the fully connected one in terms of the matrix-vector-threshold unit (MVTU) or sometimes called matrix-vector-activation unit (MVAU). But now the MVTU is preceded by a sliding window unit that produces the matrix from the input image. All of these building blocks, including the `MaxPool` layer you see in this figure, exist as templated Vitis HLS C++ functions in [finn-hlslib](https://github.com/Xilinx/finn-hlslib) and/or as RTL modules in [finn-rtllib](https://github.com/Xilinx/finn/tree/main/finn-rtllib).


To target this kind of hardware architecture with our network we'll apply a convolution lowering transformation, in addition to streamlining. You may recall the *streamlining transformation* that we applied to the TFC-w1a1 network, which is a series of mathematical simplifications that allow us to get rid of floating point scaling operations by implementing few-bit activations as thresholding operations. 

**The current implementation of streamlining is highly network-specific and may not work for your network if its topology is very different than the example network here. We hope to rectify this in future releases.**

In [ ]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_pre_post.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
# absorb final add-mul nodes 
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_streamlined.onnx")

## 3. Partitioning, Conversion to HW Layers and Folding

The next steps will be (again) very similar to what we did for the TFC-w1a1 network. We'll first convert the layers that we can put into the FPGA into their HW equivalents, separate them out into a *dataflow partition* and specialize them to HLS variants:


In [ ]:
# After this: BatchNorm → gone, some Mul → absorbed into MultiThreshold.
 
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
 
model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_pre_post.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
 
for i in range(3):
    model = model.transform(Streamline())
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    n_mul = sum(1 for n in model.graph.node if n.op_type == "Mul")
    n_bn  = sum(1 for n in model.graph.node if n.op_type == "BatchNormalization")
    print(f"  Pass {i+1}: BN={n_bn}, Mul={n_mul}")
 
model.save("finn_work_kria/step_A_streamlined_sigmoid.onnx")

  Pass 1: BN=0, Mul=1
  Pass 2: BN=0, Mul=1
  Pass 3: BN=0, Mul=1


In [ ]:
# Each Conv becomes: Transpose(NCHW→NHWC) → Im2Col → MatMul → Transpose(NHWC→NCHW)
 
model = ModelWrapper("finn_work_kria/step_A_streamlined_sigmoid.onnx")
model = model.transform(LowerConvsToMatMul())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
 
node_types = [n.op_type for n in model.graph.node]
print(f"MatMul: {node_types.count('MatMul')}")
print(f"Conv: {node_types.count('Conv')}")
print(f"Transpose: {node_types.count('Transpose')}")
print(f"Im2Col: {node_types.count('Im2Col')}")
print(f"Resize: {node_types.count('Resize')}")
print(f"Mul: {node_types.count('Mul')}")
print(f"MultiThreshold: {node_types.count('MultiThreshold')}")
 
assert node_types.count("Conv") == 0, "Some Conv nodes were not lowered!"
model.save("finn_work_kria/step_B_lowered_sigmoid.onnx")

MatMul: 27
Conv: 0  (should be 0)
Transpose: 54
Im2Col: 22
Resize: 4
Mul: 1
MultiThreshold: 28


In [ ]:
# Absorb Transposes into MultiThreshold                        

model = ModelWrapper("finn_work_kria/step_B_lowered_sigmoid.onnx")
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
 
node_types = [n.op_type for n in model.graph.node]
print(f"After AbsorbTranspose:")
print(f"  Transpose: {node_types.count('Transpose')}")
print(f"  MultiThreshold: {node_types.count('MultiThreshold')}")
model.save("finn_work_kria/step_C_absorb_transpose_sigmoid.onnx")

After AbsorbTranspose:
  Transpose: 54
  MultiThreshold: 28


In [ ]:
# Absorb remaining Mul into MultiThreshold                     

from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold
from finn.transformation.streamline.reorder import (
    MoveScalarMulPastMatMul, MoveScalarLinearPastInvariants,
)
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedMul
 
model = ModelWrapper("finn_work_kria/step_C_absorb_transpose_sigmoid.onnx")
 
for i in range(5):
    prev = sum(1 for n in model.graph.node if n.op_type == "Mul")
    model = model.transform(MoveScalarMulPastMatMul())
    model = model.transform(MoveScalarLinearPastInvariants())
    model = model.transform(CollapseRepeatedMul())
    model = model.transform(AbsorbMulIntoMultiThreshold())
    model = model.transform(Streamline())
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    curr = sum(1 for n in model.graph.node if n.op_type == "Mul")
    print(f"  Pass {i+1}: Mul {prev} -> {curr}")
    if curr == prev:
        break
 
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(RemoveUnusedTensors())
 
node_types = [n.op_type for n in model.graph.node]
print(f"\nAfter Mul absorption:")
print(f"  Mul: {node_types.count('Mul')}")
print(f"  Transpose: {node_types.count('Transpose')}")
print(f"  Resize: {node_types.count('Resize')}")
model.save("finn_work_kria/step_D_mul_absorbed_sigmoid.onnx")

  Pass 1: Mul 1 -> 1

After Mul absorption:
  Mul: 1  (target: 0 or 1)
  Transpose: 54
  Resize: 4


In [ ]:
# NHWC fix for Resize nodes, UNet specific
 
import numpy as np
from onnx import helper
 
model = ModelWrapper("finn_work_kria/step_D_mul_absorbed_sigmoid.onnx")
graph = model.graph
resize_nodes = [n for n in graph.node if n.op_type == "Resize"]
print(f"Found {len(resize_nodes)} Resize node(s) to fix")
 
def _reorder_nchw_to_nhwc(arr):
    # from NCHW to NHWC
    return np.array([arr[0], arr[2], arr[3], arr[1]], dtype=arr.dtype)
 
for rn in resize_nodes:
    inp_name = rn.input[0]
    inp_shape = model.get_tensor_shape(inp_name)
    if inp_shape is None or len(inp_shape) != 4:
        print(f"  Skipping {rn.name} -- shape {inp_shape}")
        continue
 
    # Skip if already NHWC
    layout = model.get_tensor_layout(inp_name)
    if layout == ["N", "H", "W", "C"]:
        print(f"  {rn.name}: already NHWC -- skip")
        continue
 
    nhwc_in  = inp_name + "_nhwc"
    nhwc_out = rn.output[0] + "_nhwc"
    old_out  = rn.output[0]
 
    # Transpose NCHW->NHWC before Resize
    pre = helper.make_node("Transpose", [inp_name], [nhwc_in],
                           perm=[0, 2, 3, 1], name=rn.name + "_pre")
    # Rewire Resize
    rn.input[0]  = nhwc_in
    rn.output[0] = nhwc_out
    # Transpose NHWC->NCHW after Resize
    post = helper.make_node("Transpose", [nhwc_out], [old_out],
                            perm=[0, 3, 1, 2], name=rn.name + "_post")
 
    # Reorder from NCHW to NHWC
    if len(rn.input) >= 3 and rn.input[2]:   # scales
        sc = model.get_initializer(rn.input[2])
        if sc is not None and len(sc) == 4:
            print(f"  {rn.name} scales: {sc} -> {_reorder_nchw_to_nhwc(sc)}")
            model.set_initializer(rn.input[2], _reorder_nchw_to_nhwc(sc))
    if len(rn.input) >= 4 and rn.input[3]:   # sizes
        sz = model.get_initializer(rn.input[3])
        if sz is not None and len(sz) == 4:
            print(f"  {rn.name} sizes: {sz} -> {_reorder_nchw_to_nhwc(sz)}")
            model.set_initializer(rn.input[3], _reorder_nchw_to_nhwc(sz))
 
    # Insert at positions
    idx = list(graph.node).index(rn)
    graph.node.insert(idx, pre)
    graph.node.insert(idx + 2, post)  # after
    print(f"  Wrapped {rn.name} with NCHW<->NHWC Transposes")
 
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
 
# Set layout annotation
for node in model.graph.node:
    if node.op_type == "Resize":
        model.set_tensor_layout(node.input[0], ["N", "H", "W", "C"])
 
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
 
node_types = [n.op_type for n in model.graph.node]
print(f"\nAfter NHWC fix:")
print(f"  Resize: {node_types.count('Resize')}")
print(f"  Transpose: {node_types.count('Transpose')}")
model.save("finn_work_kria/step_E_nhwc_fix_sigmoid.onnx")

Found 4 Resize node(s) to fix
  Resize_0 scales: [1. 1. 2. 2.] -> [1. 2. 2. 1.]
  Wrapped Resize_0 with NCHW<->NHWC Transposes
  Resize_1 scales: [1. 1. 2. 2.] -> [1. 2. 2. 1.]
  Wrapped Resize_1 with NCHW<->NHWC Transposes
  Resize_2 scales: [1. 1. 2. 2.] -> [1. 2. 2. 1.]
  Wrapped Resize_2 with NCHW<->NHWC Transposes
  Resize_3 scales: [1. 1. 2. 2.] -> [1. 2. 2. 1.]
  Wrapped Resize_3 with NCHW<->NHWC Transposes

After NHWC fix:
  Resize: 4
  Transpose: 62  (increased by ~8)


In [ ]:
# Absorb consecutive Transposes                                 
 
model = ModelWrapper("finn_work_kria/step_E_nhwc_fix_sigmoid.onnx")
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
 
node_types = [n.op_type for n in model.graph.node]
print(f"After AbsorbConsecutiveTransposes:")
print(f"  Transpose: {node_types.count('Transpose')}")
print(f"  Resize: {node_types.count('Resize')}")
print(f"  Total nodes: {len(node_types)}")
 
# Count which ops remain
for op in sorted(set(node_types)):
    print(f"    {op}: {node_types.count(op)}")
 
model.save("finn_work_kria/step_F_consecutive_absorbed_sigmoid.onnx")

After AbsorbConsecutiveTransposes:
  Transpose: 2  (should be much lower)
  Resize: 4
  Total nodes: 86
    Im2Col: 22
    MatMul: 27
    Mul: 1
    MultiThreshold: 28
    Resize: 4
    Sigmoid: 1
    TopK: 1
    Transpose: 2


In [ ]:
# Convert to HW layers                                         
 
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from qonnx.custom_op.registry import getCustomOp
from finn.util.basic import pynq_part_map
 
#pynq_board = "Pynq-Z2"
#fpga_part = pynq_part_map[pynq_board]
fpga_part = "xck26-sfvc784-2LV-c"
target_clk_ns = 10
 
model = ModelWrapper("finn_work_kria/step_F_consecutive_absorbed_sigmoid.onnx")
 
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferLabelSelectLayer())
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferStreamingMaxPool())
 
model = model.transform(to_hw.InferUpsample()) #important for resize
 
model = model.transform(RemoveCNVtoFCFlatten())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(InferDataLayouts())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveUnusedTensors())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
 
node_types = [n.op_type for n in model.graph.node]
print(f"After HW conversion:")
for op in sorted(set(node_types)):
    is_hw = any(k in op for k in ["MVAU", "ConvolutionInputGenerator", "Thresholding",
                                    "Upsample", "FMPadding", "Streaming", "LabelSelect", "Pool"])
    tag = "[HW]" if is_hw else "[CPU]" if op not in ("Transpose", "Reshape", "Identity") else "[struct]"
    print(f"    {op}: {node_types.count(op)}  {tag}")
 
n_transpose = node_types.count("Transpose")
print(f"\n  Transpose remaining: {n_transpose}")
 
model.save("finn_work_kria/step_G_hw_layers_sigmoid.onnx")

After HW conversion:
    ConvolutionInputGenerator: 22  [HW]
    FMPadding: 18  [HW]
    MVAU: 27  [HW]
    Mul: 1  [CPU]
    Sigmoid: 1  [CPU]
    Thresholding: 2  [HW]
    TopK: 1  [CPU]
    Transpose: 2  [struct]
    UpsampleNearestNeighbour: 4  [HW]

  Transpose remaining: 2


In [ ]:
#dataflow

from finn.util.basic import pynq_part_map

#pynq_board = "Pynq-Z2"
#fpga_part = pynq_part_map[pynq_board]
fpga_part = "xck26-sfvc784-2LV-c"
target_clk_ns = 10

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

model = ModelWrapper("finn_work_kria/step_G_hw_layers_sigmoid.onnx")

# infer tensor data layouts
model = model.transform(InferDataLayouts())
parent_model = model.transform(CreateDataflowPartition())
parent_model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_dataflow_model.onnx")

Now we have to set the *folding factors* for certain layers to adjust the performance of our accelerator, similar to the TFC-w1a1 example. We'll also set the desired FIFO depths around those layers, which are important to achieve full throughput in the accelerator.

In [15]:
# Wipe per-node codegen paths so PrepareIP regenerates everything from current folding.
import shutil, os
from qonnx.custom_op.registry import getCustomOp

CODEGEN_ATTRS = ["code_gen_dir_ipgen", "code_gen_dir_cppsim",
                 "ipgen_path", "ip_path", "executable_path"]

def purge_codegen(model):
    for node in model.graph.node:
        try:
            inst = getCustomOp(node)
        except Exception:
            continue
        for a in CODEGEN_ATTRS:
            try:
                p = inst.get_nodeattr(a)
            except Exception:
                continue
            if p and os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)
            try:
                inst.set_nodeattr(a, "")
            except Exception:
                pass
    return model

In [16]:
from finn.transformation.fpgadataflow.set_folding import SetFolding

model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_dataflow_model.onnx")

target_fps = 5
target_cycles = int(1e9 / target_clk_ns / target_fps)
print(f"Budget: {target_cycles:,} cycles/frame at {target_fps} FPS")

model = model.transform(SetFolding(target_cycles_per_frame=target_cycles))
model = model.transform(GiveUniqueNodeNames())

# Print what SetFolding chose
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
cycles = model.analysis(exp_cycles_per_layer)
if cycles:
    bottleneck = max(cycles, key=cycles.get)
    print(f"Bottleneck: {bottleneck} ({cycles[bottleneck]:,} cycles)")
    print(f"Effective FPS: {1e9 / target_clk_ns / cycles[bottleneck]:.1f}")

model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")

Budget: 20,000,000 cycles/frame at 5 FPS
Bottleneck: MVAU_hls_1 (2,359,296 cycles)
Effective FPS: 42.4


/home/teo/finn/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type FMPadding_rtl
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)
/home/teo/finn/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type UpsampleNearestNeighbour_hls
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)


In [17]:
#make child dataflow from parent
import os
m = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")

for sdp in m.get_nodes_by_op_type("StreamingDataflowPartition"):
    sdp_inst = getCustomOp(sdp)
    child_path = sdp_inst.get_nodeattr("model")
    child = ModelWrapper(child_path)
    changed = 0
    for n in child.graph.node:
        if n.op_type == "MVAU_rtl":
            n.op_type = "MVAU_hls"
            n.domain = "finn.custom_op.fpgadataflow.hls"
            changed += 1
    print(f"{child_path}: MVAU_rtl -> MVAU_hls : {changed}")
    child.save(child_path)

In [ ]:
#bottleneck to _hls
from qonnx.custom_op.registry import getCustomOp

m = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")
RTL_TO_HLS = {
    "MVAU_rtl": ("MVAU_hls", "finn.custom_op.fpgadataflow.hls"),
}
changed = {}
for n in m.graph.node:
    if n.op_type in RTL_TO_HLS:
        new_op, new_dom = RTL_TO_HLS[n.op_type]
        changed[n.op_type] = changed.get(n.op_type, 0) + 1
        n.op_type = new_op
        n.domain = new_dom
print(changed)
m.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")

{'MVAU_rtl': 1}


Our network is now ready and we can start with the hardware generation.

## 4. Hardware Generation

From this point onward, the steps we have to follow do not depend on the particular network and will be exactly the same as the TFC-w1a1 example. **which may take about 30 minutes depending on your host computer**. For more details about what's going on in this step, please consult the [TFC end-to-end notebook](tfc_end2end_example.ipynb) or the appropriate section in the [FINN documentation](https://finn.readthedocs.io/en/latest/hw_build.html).

In [ ]:
#check
from qonnx.custom_op.registry import getCustomOp
m = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")
print("parent ops:", {n.op_type for n in m.graph.node})

for sdp in m.get_nodes_by_op_type("StreamingDataflowPartition"):
    child = ModelWrapper(getCustomOp(sdp).get_nodeattr("model"))
    types = [n.op_type for n in child.graph.node]
    print("child counts:", {t: types.count(t) for t in set(types)})

parent ops: {'ConvolutionInputGenerator_rtl', 'Thresholding_rtl', 'MVAU_hls', 'FMPadding_rtl', 'UpsampleNearestNeighbour_hls'}


In [ ]:

# SIMD

import math
from qonnx.custom_op.registry import getCustomOp

def _min_divisor_ge(n, lo):
    return next(d for d in range(lo, n + 1) if n % d == 0)

def enforce_mvau_hls_folding(model):
    changes = []
    for node in model.graph.node:
        if not node.op_type.startswith("MVAU"):
            continue
        inst = getCustomOp(node)
        for dim_attr, fold_attr in (("MW", "SIMD"), ("MH", "PE")):
            dim = inst.get_nodeattr(dim_attr)
            fold = inst.get_nodeattr(fold_attr)
            need = max(1, math.ceil(dim / 1024))
            if fold >= need:
                continue
            new = _min_divisor_ge(dim, need)
            inst.set_nodeattr(fold_attr, new)
            changes.append((node.name, dim_attr, dim, fold_attr, fold, new))
    return model, changes

model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")
model, changes = enforce_mvau_hls_folding(model)
for c in changes:
    print("  {}: {}={}, {} {} -> {}".format(*c))
model = model.transform(GiveUniqueNodeNames())
model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")

  MVAU_hls_13: MW=1152, SIMD 1 -> 2


In [ ]:
# InsertDWC (data width converters between layers)

from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.transformation.general import GiveUniqueNodeNames

model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")
model = model.transform(InsertDWC())
model = model.transform(SpecializeLayers(fpga_part))
model = model.transform(GiveUniqueNodeNames())

n_dwc = sum(1 for n in model.graph.node if "DataWidthConverter" in n.op_type)
print(f"Inserted {n_dwc} DWC nodes")
model.save("finn_work_kria/step_S2_dwc_sigmoid.onnx")

Inserted 45 DWC nodes


In [ ]:
# InsertFIFO

from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO

model = ModelWrapper("finn_work_kria/step_S2_dwc_sigmoid.onnx")
model = model.transform(InsertFIFO(create_shallow_fifos=True))
model = model.transform(SpecializeLayers(fpga_part))
model = model.transform(GiveUniqueNodeNames())

n_fifo = sum(1 for n in model.graph.node if "FIFO" in n.op_type)
print(f"Inserted {n_fifo} FIFOs")
model.save("finn_work_kria/step_S3_fifo_sigmoid.onnx")

Inserted 119 FIFOs


In [ ]:

# PrepareIP (generate HLS/RTL code for each node)

from finn.transformation.fpgadataflow.prepare_ip import PrepareIP

model = ModelWrapper("finn_work_kria/step_S3_fifo_sigmoid.onnx")
print("Generating IP code for all nodes ...")
model = model.transform(PrepareIP(fpga_part, target_clk_ns))
print("PrepareIP done")
model.save("finn_work_kria/step_S4_prepared_sigmoid.onnx")

Generating IP code for all nodes ...
PrepareIP done


In [ ]:
# HLSSynthIP (run Vitis HLS on each HLS node)

from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP

model = ModelWrapper("finn_work_kria/step_S4_prepared_sigmoid.onnx")
print("Running HLS synthesis ...")
model = model.transform(HLSSynthIP())
print("HLS synthesis done")
model.save("finn_work_kria/step_S5_hlssynth_sigmoid.onnx")

Running HLS synthesis ...
HLS synthesis done


In [28]:
#stitched ip after hls synth

from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP

model = ModelWrapper("finn_work_kria/step_S5_hlssynth_sigmoid.onnx")
model = model.transform(CreateStitchedIP(fpga_part, target_clk_ns, vitis=False))
model.save("finn_work_kria/step_S6_stitched_ip_sigmoid.onnx")

proj = model.get_metadata_prop("vivado_stitch_proj")
print("stitch project:", proj)
print("packaged IP    :", proj + "/ip")

stitch project: /home/teo/finn_build/vivado_stitch_proj_pzki_y8p
packaged IP    : /home/teo/finn_build/vivado_stitch_proj_pzki_y8p/ip


In [ ]:
####resurse#####
from finn.analysis.fpgadataflow.hls_synth_res_estimation import hls_synth_res_estimation

model = ModelWrapper("finn_work_kria/step_S5_hlssynth_sigmoid.onnx")
res = model.analysis(hls_synth_res_estimation)

# KV260 = Zynq UltraScale+ XCK26
BOARD = {"LUT": 117120, "FF": 234240, "BRAM_18K": 288, "URAM": 64, "DSP48E": 1248}

total = {k: 0 for k in BOARD}
for name, r in res.items():
    for k in total:
        total[k] += int(r.get(k, 0))

print(f"{'Resource':<10} {'Used':>8} {'Avail':>8} {'%':>7}")
print("-" * 38)
for k, avail in BOARD.items():
    pct = 100 * total[k] / avail
    tag = "OK" if pct < 85 else "TIGHT" if pct < 100 else "OVER"
    print(f"{k:<10} {total[k]:>8} {avail:>8} {pct:>6.1f}%  {tag}")

In [ ]:
# stale codegen wipe
import shutil, glob
for d in glob.glob("/tmp/finn_dev_teo/*"):
    shutil.rmtree(d, ignore_errors=True)

# clear cached codegen paths on nodes too
from qonnx.custom_op.registry import getCustomOp
m = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")
for n in m.graph.node:
    try: inst = getCustomOp(n)
    except: continue
    for a in ["code_gen_dir_ipgen","code_gen_dir_cppsim","ipgen_path","ip_path","executable_path"]:
        try: inst.set_nodeattr(a, "")
        except: pass
m.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")

Sinteza efectiva

In [23]:
import os
from finn.builder.build_dataflow_config import DataflowBuildConfig

os.environ["NUM_DEFAULT_WORKERS"] = "2"  # max parallel HLS syntheses
os.environ["VIVADO_SYNTH_STRATEGY"] = "Flow_RuntimeOptimised"

# Register KV260 (Zynq UltraScale+ XCK26) in FINN's board maps
from finn.util.basic import pynq_part_map
try:
    from finn.util.basic import pynq_native_port_width_map
except ImportError:
    pynq_native_port_width_map = None

PLATFORM = "KV260_SOM"
pynq_part_map[PLATFORM] = "xck26-sfvc784-2LV-c"
if pynq_native_port_width_map is not None:
    pynq_native_port_width_map.setdefault(PLATFORM, 128)  # HP port width

from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

model = ModelWrapper("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_folded.onnx")

from qonnx.custom_op.registry import getCustomOp
for n in model.graph.node:
    if n.op_type.startswith("MVAU") or "MatrixVectorActivation" in n.op_type:
        try: getCustomOp(n).set_nodeattr("preferred_impl_style", "hls")
        except: pass

print("Running ZynqBuild ...")
model = model.transform(ZynqBuild(platform=PLATFORM, period_ns=target_clk_ns))
model.save("finn_work_kria/step_S6_zynqbuild.onnx")

from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
model = model.transform(MakePYNQDriver("zynq-iodma"))   # correct platform for Zynq US+
model.save("finn_work_kria/step_S7_driver.onnx")
print("bitfile:", model.get_metadata_prop("bitfile"))

Running ZynqBuild ...


/home/teo/finn/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 120 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(
/home/teo/finn/src/finn/transformation/fpgadataflow/insert_fifo.py:234: UserWarning: Input FIFO for IODMA_hls_0_out0 has depth 2 and won't
                        be created. This may cause RTL simulation issues.
                        
  warnings.warn(
/home/teo/finn/src/finn/transformation/fpgadataflow/insert_fifo.py:294: UserWarning: Output FIFO for MVAU_hls_26_out0 has depth 2 and won't
                        be created. This may cause RTL simulation issues.
                        
  warnings.warn(
/home/teo/finn/src/finn/transformation/fpgadataflow/create_stitched_ip.py:290: UserWarning: First node is not StreamingFIFO or IODMA.
                You may experience incorrect stitched-IP rtlsim or hardware
                behavior. It is strongly recommended to insert FIFOs prior to
                calling CreateSt

bitfile: /tmp/finn_dev_teo/vivado_zynq_proj_k_881256/resizer.bit


In [24]:
##for debug##

LOG_ID = "k_881256"

import glob
d = f"/tmp/finn_dev_teo/vivado_zynq_proj_{LOG_ID}"
out = f"vivado_logs_{LOG_ID}.txt"

with open(out, "w") as w:
    for f in sorted(glob.glob(f"{d}/**/*.log", recursive=True) + glob.glob(f"{d}/**/*.rpt", recursive=True)):
        w.write(f"\n===== {f} =====\n")
        w.write(open(f, errors="replace").read())

print("saved ->", out)

saved -> vivado_logs_k_881256.txt


In [26]:
model.save("finn_work_kria/quant_seq_unet_W4A4_b128_sigmoid_synth.onnx")

After the `ZynqBuild` we run one additional transformation to generate a PYNQ driver for the accelerator.

## 5. Deployment and Execution

The bitfile and generated driver files(s) will be copied into a deployment folder which then can be used to run the network on the PYNQ board.

In [27]:
from shutil import copy
from distutils.dir_util import copy_tree

# create directory for deployment files
deployment_dir = make_build_dir(prefix="pynq_deployment_")
model.set_metadata_prop("pynq_deployment_dir", deployment_dir)

# get and copy necessary files
# .bit and .hwh file
bitfile = model.get_metadata_prop("bitfile")
hwh_file = model.get_metadata_prop("hw_handoff")
deploy_files = [bitfile, hwh_file]

for dfile in deploy_files:
    if dfile is not None:
        copy(dfile, deployment_dir)

# driver.py and python libraries
pynq_driver_dir = model.get_metadata_prop("pynq_driver_dir")
copy_tree(pynq_driver_dir, deployment_dir)

['/tmp/finn_dev_teo/pynq_deployment_kln4tku1/validate.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/driver_base.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/qonnx/core/datatype.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/qonnx/core/__init__.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/qonnx/util/basic.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/qonnx/util/__init__.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/finn/util/data_packing.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/finn/util/__init__.py',
 '/tmp/finn_dev_teo/pynq_deployment_kln4tku1/driver.py']

In [28]:
! ls {deployment_dir}

driver_base.py	finn   resizer.bit  runtime_weights
driver.py	qonnx  resizer.hwh  validate.py


In [29]:
from shutil import make_archive
make_archive('deploy-on-kria', 'zip', deployment_dir)

'/home/teo/finn/notebooks/deploy-on-kria.zip'